# NeuralGCM lead-time sweep — Aug 2025 St. John's heat wave

**Question:** can NeuralGCM reproduce a real observed heat wave, and from how far ahead?
We initialise the 2.8° deterministic model from ERA5 at a range of lead times before the
event and integrate each forecast to the event, then measure how well each captures the
heat wave at St. John's. The **crossover lead time** (where forecasts start to reliably
capture it) is the deliverable.

**Event — St. John's Intl A (ECCC 8403505) observed daily Tmax (°C):**
Aug 9: 30.5 · 10: 27.7 · 11: 28.1 · **12: 30.9 (peak)** · 13: 30.3 · 14: 27.8 · 15: 29.2.
Hot stretch Aug 9–15, then a crash to 13.0 °C on Aug 16 — the Avalon's longest-ever (7-day) heat warning.

### Before you Run all
1. **Runtime ▸ Change runtime type ▸ GPU** (T4 is fine; pick **High-RAM** if available).
2. Run all. With the defaults (`DATA_INNER_HOURS=24`, `CHUNK_DAYS=5`) peak RAM is ~4–7 GB and it runs on a standard GPU runtime.
3. On a **High-RAM** runtime, set `DATA_INNER_HOURS=6`, `CHUNK_DAYS=3` for a true daily-**max** Tmax proxy.

Pipeline mirrors `climate_sim_neuralgcm_stjohns-2020.ipynb`: ERA5 from ARCO-ERA5 (GCS) → conservative regrid 0.25°→2.8° (block-chunked) → `encode` IC → `unroll` forward.

In [ ]:
!pip install -q -U neuralgcm dinosaur gcsfs

In [ ]:
import gc, jax
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import gcsfs, pickle, neuralgcm
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils

print("JAX devices:", jax.devices())
if not any(d.platform in ("gpu", "cuda") for d in jax.devices()):
    print("\u26a0\ufe0f  No GPU detected \u2014 Runtime \u25b8 Change runtime type \u25b8 GPU, then Run all.")

# Plot style. NOTE: the repo's scripts/common.py uses LaTeX (text.usetex), which
# usually isn't installed on Colab and would crash plotting \u2014 so we use a light
# non-LaTeX style here. Re-render final figures locally with common.mpl_apply().
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.titlesize": 12})

## Config

In [ ]:
MODEL_NAME = "v1/deterministic_2_8_deg.pkl"   # same checkpoint as the 2020 run
ERA5_PATH  = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

# Observed St. John's Intl A daily Tmax (\u00b0C) \u2014 ground truth for the overlay.
OBS_TMAX = {"2025-08-09": 30.5, "2025-08-10": 27.7, "2025-08-11": 28.1,
            "2025-08-12": 30.9, "2025-08-13": 30.3, "2025-08-14": 27.8,
            "2025-08-15": 29.2}
EVENT_PEAK  = np.datetime64("2025-08-12")   # station peak (30.9 \u00b0C)
EVENT_START = np.datetime64("2025-08-09")
EVENT_END   = np.datetime64("2025-08-15")

# Lead times (days before the peak) at which to initialise.
LEAD_TIMES_DAYS = [20, 18, 16, 14, 12, 10, 9, 7, 5, 3]

# Memory knobs (see notes at top). NeuralGCM needs the GLOBAL ERA5 field, so peak
# RAM = one regrid block at 0.25\u00b0 (~0.77 GB/snapshot, all 37 levels).
DATA_INNER_HOURS = 24      # 24 = one snapshot/day; 6 = 6-hourly (needs High-RAM)
CHUNK_DAYS       = 5

STJOHNS_LAT = 47.5615
STJOHNS_LON = 360.0 - 52.7126   # \u2248 307.29 \u00b0E (0\u2013360 convention)
RNG_SEED = 42

from pathlib import Path
DATA_DIR = Path("data"); PLOTS_DIR = Path("plots")
DATA_DIR.mkdir(exist_ok=True); PLOTS_DIR.mkdir(exist_ok=True)

## Load model, open ERA5, build regridder

In [ ]:
print(f"Loading model {MODEL_NAME} \u2026")
gcs = gcsfs.GCSFileSystem(token="anon")
with gcs.open(f"gs://neuralgcm/models/{MODEL_NAME}", "rb") as f:
    ckpt = pickle.load(f)
model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)
print("input_variables:", model.input_variables)

In [ ]:
print("Opening ARCO-ERA5 (lazy) \u2026")
full_era5 = xr.open_zarr(ERA5_PATH, chunks=None, storage_options=dict(token="anon"))

era5_grid = spherical_harmonic.Grid(
    latitude_nodes=full_era5.sizes["latitude"],
    longitude_nodes=full_era5.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(full_era5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(full_era5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(
    era5_grid, model.data_coords.horizontal, skipna=True)

# Shift forcings (SST / sea-ice) by 24 h, as the model expects.
shifted_full = full_era5[model.input_variables + model.forcing_variables].pipe(
    xarray_utils.selective_temporal_shift,
    variables=model.forcing_variables, time_shift="24 hours")

## Load + regrid the window (block-chunked to bound RAM)

In [ ]:
def load_regridded(t0, t1, inner_hours, chunk_days):
    sub = (shifted_full.sel(time=slice(t0, t1))
           .isel(time=slice(None, None, inner_hours)))          # lazy
    steps = max(1, chunk_days * (24 // inner_hours))
    n = sub.sizes["time"]; pieces = []
    for i in range(0, n, steps):
        chunk = sub.isel(time=slice(i, i + steps)).compute()     # one block at 0.25\u00b0
        g = xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(chunk, regridder))
        pieces.append(g)
        print(f"  block {i // steps + 1}: {chunk.sizes['time']:3d} snapshots regridded")
        del chunk; gc.collect()
    return xr.concat(pieces, dim="time")

buffer = np.timedelta64(2, "D")
window_start = EVENT_PEAK - np.timedelta64(max(LEAD_TIMES_DAYS), "D")
window_end   = EVENT_END + buffer
print(f"Loading + regridding ERA5 {window_start} \u2192 {window_end} \u2026")
eval_era5 = load_regridded(window_start, window_end, DATA_INNER_HOURS, CHUNK_DAYS)
print(f"\u2192 {eval_era5.sizes['time']} snapshots on the model grid.")

In [ ]:
def daily_max_t1000(ds):
    """St. John's nearest-grid-point 1000 hPa temperature (\u00b0C), daily MAX."""
    pt = ds.temperature.sel(level=1000).sel(
        latitude=STJOHNS_LAT, longitude=STJOHNS_LON, method="nearest") - 273.15
    return pt.resample(time="1D").max().to_pandas()

era5_truth = daily_max_t1000(eval_era5)     # ERA5 on the model grid = fair reference
print(era5_truth.loc["2025-08-07":"2025-08-16"].round(1))

## Forecast sweep: one deterministic unroll per lead time

In [ ]:
def run_forecast(init_time):
    fc = eval_era5.sel(time=slice(init_time, window_end))
    n_steps = fc.sizes["time"]                         # start_with_input=True
    inputs   = model.inputs_from_xarray(fc.isel(time=0))
    forcing0 = model.forcings_from_xarray(fc.isel(time=0))
    state    = model.encode(inputs, forcing0, jax.random.key(RNG_SEED))
    all_forcings = model.forcings_from_xarray(fc)      # perfect (ERA5) forcings
    td = np.timedelta64(1, "h") * DATA_INNER_HOURS
    _, preds = model.unroll(state, all_forcings, steps=n_steps,
                            timedelta=td, start_with_input=True)
    ds = model.data_to_xarray(
        preds, times=np.arange(n_steps) * DATA_INNER_HOURS).as_numpy()
    ds = ds.assign_coords(time=fc.time.values)
    return daily_max_t1000(ds)

records, traj = [], {}
win = slice(pd.Timestamp(EVENT_START), pd.Timestamp(EVENT_END))
for lead in LEAD_TIMES_DAYS:
    init = EVENT_PEAK - np.timedelta64(lead, "D")
    print(f"lead {lead:2d} d  (init {init}) \u2026")
    s = run_forecast(init); traj[lead] = s
    f, o = s.loc[win], era5_truth.loc[win]
    c = f.index.intersection(o.index); f, o = f.loc[c], o.loc[c]
    records.append({"lead_days": lead, "init_date": pd.Timestamp(init).date(),
                    "fc_peak_C": round(float(f.max()), 2),
                    "era5_peak_C": round(float(o.max()), 2),
                    "peak_err_C": round(float(f.max() - o.max()), 2),
                    "window_rmse_C": round(float(np.sqrt(((f - o) ** 2).mean())), 2)})
    print("   peak err {:+.2f} \u00b0C   window RMSE {:.2f} \u00b0C".format(
        records[-1]["peak_err_C"], records[-1]["window_rmse_C"]))
    gc.collect()

skill = pd.DataFrame(records).sort_values("lead_days")
skill

## Save outputs

In [ ]:
skill.to_csv(DATA_DIR / "heatwave_leadtime_skill_aug2025.csv", index=False)
traj_da = xr.concat(
    [xr.DataArray(s.values, coords={"time": s.index}, dims="time") for s in traj.values()],
    dim=pd.Index(list(traj.keys()), name="lead_days"))
xr.Dataset({"tmax_proxy_C": traj_da,
            "era5_tmax_proxy_C": xr.DataArray(
                era5_truth.values, coords={"time": era5_truth.index}, dims="time")}
          ).to_netcdf(DATA_DIR / "heatwave_leadtime_stjohns_aug2025.nc")
print("Saved \u2192 data/heatwave_leadtime_skill_aug2025.csv + .nc")

## Plots

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(era5_truth.index, era5_truth.values, "k", lw=2.5, label="ERA5 (truth)", zorder=10)
obs = pd.Series({pd.Timestamp(k): v for k, v in OBS_TMAX.items()}).sort_index()
ax.scatter(obs.index, obs.values, c="red", s=45, zorder=11, label="station obs Tmax")
for c, lead in zip(plt.cm.viridis(np.linspace(0, 1, len(LEAD_TIMES_DAYS))), LEAD_TIMES_DAYS):
    s = traj[lead]; ax.plot(s.index, s.values, color=c, lw=1.3, label=f"init \u2212{lead} d")
ax.axvspan(pd.Timestamp(EVENT_START), pd.Timestamp(EVENT_END), color="red", alpha=0.08)
ax.set_ylabel("St. John's daily-max T$_{1000}$ (\u00b0C)")
ax.set_title("NeuralGCM forecasts of the Aug 2025 St. John's heat wave by lead time")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "heatwave_leadtime_trajectories.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
a1.axhline(0, color="grey", ls="--"); a1.plot(skill.lead_days, skill.peak_err_C, "o-")
a1.set(xlabel="lead time (days before peak)", ylabel="peak Tmax error (\u00b0C)  fc\u2212ERA5",
       title="Peak error vs lead time"); a1.invert_xaxis()
a2.plot(skill.lead_days, skill.window_rmse_C, "s-", color="C3")
a2.set(xlabel="lead time (days before peak)", ylabel="event-window RMSE (\u00b0C)",
       title="Event-window RMSE vs lead time"); a2.invert_xaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "heatwave_leadtime_skill.pdf", bbox_inches="tight")
plt.show()
print(skill.to_string(index=False))

## Download outputs (Colab)

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("heatwave_leadtime_outputs", "zip", ".", "data")
    shutil.make_archive("heatwave_leadtime_plots", "zip", ".", "plots")
    files.download("heatwave_leadtime_outputs.zip")
    files.download("heatwave_leadtime_plots.zip")
except Exception as e:
    print("Not on Colab / download skipped:", e)

## Reading the result

- **Crossover:** the lead time where `peak_err_C` and `window_rmse_C` collapse toward zero is when the heat wave becomes "baked into" the initial conditions. Long leads (~20 d) are expected to miss it (past the deterministic predictability limit).
- **Single-run caveat:** these are deterministic forecasts \u2014 one run per lead. A long-lead miss could be luck, not true unpredictability. Confirm the crossover with a small ensemble (perturbed ICs or the stochastic checkpoint) before trusting it.
- **Tmax proxy:** daily max of 1000 hPa T (the 2.8\u00b0 model has no true 2 m Tmax). Verification is on the model grid vs ERA5; the red points show the station Tmax for context.
- **Next step:** the crossover lead time defines the window in which the worst-case initial-condition optimisation is meaningful.